# Phase 3 - Machine Learning Marketing

**Objectif** : Construire 4 modèles ML exploitables par les équipes marketing :
1. **K-Means** : Segmentation comportementale des clients
2. **Linear Regression** : Impact des campagnes marketing sur les ventes
3. **Random Forest** : Identification des drivers de vente

> Base : tables `ANALYTICS` construites en Phase 3.

In [ ]:
USE DATABASE anycompany_lab;
USE SCHEMA ANALYTICS;

---
## 1. Préparation des données ML

In [ ]:
# Chargement des features clients depuis Snowflake
import snowflake.snowpark as snowpark
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, classification_report,
    confusion_matrix, mean_squared_error, r2_score
)
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBClassifier
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# Récupération de la session Snowpark active
session = get_active_session()

# Chargement de CUSTOMER_FEATURES
df_cf = session.table("ANYCOMPANY_LAB.ANALYTICS.CUSTOMER_FEATURES").to_pandas()
print(f"CUSTOMER_FEATURES : {df_cf.shape[0]} lignes, {df_cf.shape[1]} colonnes")
df_cf.head()

In [ ]:
print(f"Compte : {session.get_current_account()}")
print(f"Rôle actuel : {session.get_current_role()}")
print(f"Entrepôt (Warehouse) : {session.get_current_warehouse()}")
print(f"Base de données : {session.get_current_database()}")
print(f"Schéma : {session.get_current_schema()}")

In [ ]:
# Chargement de VENTE_ENRICHIES (agrégat mensuel)
df_sales_raw = session.table("ANYCOMPANY_LAB.ANALYTICS.VENTE_ENRICHIES").to_pandas()

# Agrégation mensuelle par région
df_sales = (
    df_sales_raw
    .groupby(["REGION", "YEAR", "MONTH", "PROMOTION_FLAG", "MARKETING_FLAG"])
    .agg(
        TOTAL_SALES=("AMOUNT", "sum"),
        NB_TRANSACTIONS=("TRANSACTION_ID", "count"),
        AVG_DISCOUNT=("DISCOUNT_PERCENTAGE", "mean"),
        AVG_BUDGET=("BUDGET", "mean"),
        AVG_CONVERSION=("CONVERSION_RATE", "mean")
    )
    .reset_index()
    .fillna(0)
)
print(f"Données ventes agrégées : {df_sales.shape}")
df_sales.head()

In [ ]:
# Chargement de PROMOTION_ACTIVES
df_promos = session.table("ANYCOMPANY_LAB.ANALYTICS.PROMOTION_ACTIVES").to_pandas()
print(f"PROMOTION_ACTIVES : {df_promos.shape}")
df_promos.head()

## 2. Modèle 1 - K-Means : Segmentation comportementale

**Objectif** : Identifier des groupes de clients homogènes (profil d'achat, usage des promos, budget dépensé).

**Features utilisées** : `NOMBRE_D_ACHAT`, `TOTAL_DEPENSE`, `PANIER_MOYEN`, `PROMOTION_USAGE`, `TRANCHE_AGE`, `TRANCHE_REVENUS`.

In [ ]:
# === PRÉPARATION K-MEANS ===

# Encodage des variables catégorielles
le_age = LabelEncoder()
le_rev = LabelEncoder()
le_reg = LabelEncoder()

df_km = df_cf.copy()
df_km["AGE_ENC"]    = le_age.fit_transform(df_km["TRANCHE_AGE"].fillna("Unknown"))
df_km["REV_ENC"]    = le_rev.fit_transform(df_km["TRANCHE_REVENUS"].fillna("Unknown"))
df_km["REGION_ENC"] = le_reg.fit_transform(df_km["REGION"].fillna("Unknown"))

# Features numériques pour le clustering
features_km = [
    "NOMBRE_D_ACHAT", "TOTAL_DEPENSE",
    "PANIER_MOYEN", "PROMOTION_USAGE",
    "AGE_ENC", "REV_ENC"
]

X_km = df_km[features_km].fillna(0)

# Normalisation
scaler_km = StandardScaler()
X_km_scaled = scaler_km.fit_transform(X_km)

print("Features K-Means :", features_km)
print(f"Matrice d'entrée : {X_km_scaled.shape}")

In [ ]:
# === MÉTHODE DU COUDE — Choix optimal de K ===
import matplotlib.pyplot as plt
from joblib import parallel_backend

inertias = []
sil_scores = []
K_range = range(2, 9)


for k in K_range:

    with parallel_backend('threading', n_jobs=1):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        km.fit(X_km_scaled)
        inertias.append(km.inertia_)
        sil_scores.append(silhouette_score(X_km_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(K_range, inertias, marker='o', color='steelblue')
ax1.set_title('Méthode du coude (Inertie)')
ax1.set_xlabel('Nombre de clusters K')
ax1.set_ylabel('Inertie')
ax1.grid(True, alpha=0.3)

ax2.plot(K_range, sil_scores, marker='s', color='coral')
ax2.set_title('Score Silhouette par K')
ax2.set_xlabel('Nombre de clusters K')
ax2.set_ylabel('Silhouette Score')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/kmeans_elbow.png', dpi=100, bbox_inches='tight')
plt.show()

best_k = K_range[sil_scores.index(max(sil_scores))]
print(f"K optimal (silhouette max) : {best_k} — Score : {max(sil_scores):.4f}")

In [ ]:
# === ENTRAÎNEMENT K-MEANS FINAL ===
K_FINAL = best_k  # ou forcer K_FINAL = 4 si métier le requiert

kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=20)
df_km["SEGMENT"] = kmeans.fit_predict(X_km_scaled)

# Profil de chaque segment
segment_profile = (
    df_km.groupby("SEGMENT")[features_km + ["TRANCHE_AGE", "TRANCHE_REVENUS"]]
    .agg({
        "NOMBRE_D_ACHAT":  "mean",
        "TOTAL_DEPENSE":   "mean",
        "PANIER_MOYEN":    "mean",
        "PROMOTION_USAGE": "mean",
        "TRANCHE_AGE":     lambda x: x.value_counts().index[0],
        "TRANCHE_REVENUS": lambda x: x.value_counts().index[0]
    })
    .round(2)
    .rename(columns={
        "NOMBRE_D_ACHAT":  "nb_achats_moy",
        "TOTAL_DEPENSE":   "depense_moy",
        "PANIER_MOYEN":    "panier_moy",
        "PROMOTION_USAGE": "usage_promo_moy",
        "TRANCHE_AGE":     "age_dominant",
        "TRANCHE_REVENUS": "revenu_dominant"
    })
)
print("Profil des segments :")
print(segment_profile)

# Nommage métier automatique (heuristique)
def label_segment(row):
    if row["depense_moy"] > segment_profile["depense_moy"].quantile(0.75):
        return "Champions"
    elif row["usage_promo_moy"] > segment_profile["usage_promo_moy"].median():
        return "Chasseurs de promos"
    elif row["nb_achats_moy"] < segment_profile["nb_achats_moy"].quantile(0.25):
        return "Dormants"
    else:
        return "Clients réguliers"

segment_profile["label_metier"] = segment_profile.apply(label_segment, axis=1)
print("\nLabels métier assignés :")
print(segment_profile[["label_metier"]])

In [ ]:
# === EXPORT VERS SNOWFLAKE ===
# Mapping segment_id → label métier
label_map = segment_profile["label_metier"].to_dict()

df_segments_out = df_km[["CUSTOMER_ID", "SEGMENT", "REGION", "TRANCHE_AGE", "TRANCHE_REVENUS"]].copy()
df_segments_out["SEGMENT_LABEL"] = df_segments_out["SEGMENT"].map(label_map)
df_segments_out = df_segments_out.rename(columns={
    "CUSTOMER_ID":    "CUSTOMER_ID",
    "SEGMENT":        "SEGMENT_ID",
    "SEGMENT_LABEL":  "SEGMENT_NAME"
})

# Upload Snowflake
df_sp = session.create_dataframe(df_segments_out)
df_sp.write.mode("overwrite").save_as_table("ANYCOMPANY_LAB.ANALYTICS.CUSTOMER_SEGMENTS")

print(f"Table CUSTOMER_SEGMENTS écrite : {len(df_segments_out)} lignes")
print(df_segments_out["SEGMENT_NAME"].value_counts())

## 3. Modèle 3 - Régression Linéaire : Impact des campagnes marketing

**Objectif** : Quantifier l'effet des campagnes marketing et promotions sur le CA (uplift).

**Variable cible** : `TOTAL_SALES` (CA mensuel par région)

**Features** : budget campagne, taux de conversion, durée promo, discount, flags promo/campaign.

In [ ]:
# === PRÉPARATION RÉGRESSION ===
df_reg = df_sales.copy()

# Encodage région
df_reg["REGION_ENC"] = LabelEncoder().fit_transform(df_reg["REGION"].fillna("Unknown"))

features_reg = [
    "PROMOTION_FLAG", "MARKETING_FLAG",
    "AVG_DISCOUNT", "AVG_BUDGET", "AVG_CONVERSION",
    "NB_TRANSACTIONS", "REGION_ENC",
    "MONTH"
]

X3 = df_reg[features_reg].fillna(0)
y3 = df_reg["TOTAL_SALES"].fillna(0)

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=42
)

print(f"Dataset régression : {X3.shape}")
print(f"CA moyen : {y3.mean():,.0f} | min : {y3.min():,.0f} | max : {y3.max():,.0f}")

In [ ]:
# === ENTRAÎNEMENT RÉGRESSION LINÉAIRE ===
from sklearn.preprocessing import StandardScaler

scaler_reg = StandardScaler()
X3_train_sc = scaler_reg.fit_transform(X3_train)
X3_test_sc  = scaler_reg.transform(X3_test)

reg_model = LinearRegression()
reg_model.fit(X3_train_sc, y3_train)

y3_pred = reg_model.predict(X3_test_sc)

rmse = np.sqrt(mean_squared_error(y3_test, y3_pred))
r2   = r2_score(y3_test, y3_pred)

print(f"=== Régression Linéaire ===")
print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:,.0f}")

# Coefficients
coef_df = pd.DataFrame({
    "Feature": features_reg,
    "Coefficient": reg_model.coef_
}).sort_values("Coefficient", ascending=False)

print("\nCoefficients (impact standardisé sur le CA) :")
print(coef_df.to_string(index=False))

In [ ]:
# === VISUALISATION — Réel vs Prédit + Coefficients ===
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Réel vs Prédit
ax1.scatter(y3_test, y3_pred, alpha=0.5, color='steelblue', s=20)
lim = max(y3_test.max(), y3_pred.max())
ax1.plot([0, lim], [0, lim], 'r--', lw=1)
ax1.set_xlabel('CA réel')
ax1.set_ylabel('CA prédit')
ax1.set_title(f'Régression — Réel vs Prédit\nR²={r2:.3f} | RMSE={rmse:,.0f}')
ax1.grid(True, alpha=0.3)

# Coefficients
colors = ['coral' if c > 0 else 'steelblue' for c in coef_df["Coefficient"]]
ax2.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors)
ax2.axvline(x=0, color='black', lw=0.8)
ax2.set_title('Impact des features sur le CA (β standardisés)')
ax2.set_xlabel('Coefficient')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('/tmp/regression_results.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# === CALCUL UPLIFT PROMO/CAMPAGNE PAR RÉGION ===
df_reg_out = df_reg.copy()
X3_full_sc = scaler_reg.transform(df_reg[features_reg].fillna(0))
df_reg_out["PREDICTED_SALES"] = reg_model.predict(X3_full_sc).round(2)

# Uplift = vente prédite avec promo - sans promo (simulation)
df_no_promo = df_reg.copy()
df_no_promo["PROMOTION_FLAG"] = 0
df_no_promo["CAMPAIGN_FLAG"]  = 0
df_no_promo["AVG_DISCOUNT"]   = 0

X_no_promo_sc = scaler_reg.transform(df_no_promo[features_reg].fillna(0))
df_reg_out["BASELINE_SALES"]  = reg_model.predict(X_no_promo_sc).round(2)
df_reg_out["UPLIFT"]          = (df_reg_out["PREDICTED_SALES"] - df_reg_out["BASELINE_SALES"]).round(2)
df_reg_out["UPLIFT_PCT"]      = ((df_reg_out["UPLIFT"] / df_reg_out["BASELINE_SALES"].replace(0, np.nan)) * 100).round(2)

uplift_summary = (
    df_reg_out.groupby("REGION")[["UPLIFT", "UPLIFT_PCT", "TOTAL_SALES", "PREDICTED_SALES"]]
    .mean().round(2)
)
print("Uplift moyen par région :")
print(uplift_summary)

# Export Snowflake
df_sp3 = session.create_dataframe(df_reg_out)
df_sp3.write.mode("overwrite").save_as_table("ANYCOMPANY_LAB.ANALYTICS.ML_PROMOTION_FEATURES")
print(f"\nTable ML_PROMOTION_FEATURES écrite : {len(df_reg_out)} lignes")

## 4. Modèle 4 - Random Forest : Drivers de vente

**Objectif** : Identifier les variables les plus influentes sur les ventes, toutes choses égales par ailleurs.

**Variable cible** : `TOTAL_SALES`

**Avantage** vs régression : capture les non-linéarités et interactions entre variables.

In [ ]:
# === ENTRAÎNEMENT RANDOM FOREST ===
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X3_train, y3_train)

y3_rf_pred = rf_model.predict(X3_test)
rmse_rf    = np.sqrt(mean_squared_error(y3_test, y3_rf_pred))
r2_rf      = r2_score(y3_test, y3_rf_pred)

print(f"=== Random Forest ===")
print(f"R²   : {r2_rf:.4f}")
print(f"RMSE : {rmse_rf:,.0f}")

# Comparaison
print(f"\n--- Comparaison ---")
print(f"Régression Linéaire → R²={r2:.4f} | RMSE={rmse:,.0f}")
print(f"Random Forest       → R²={r2_rf:.4f} | RMSE={rmse_rf:,.0f}")

In [ ]:
# === FEATURE IMPORTANCE RANDOM FOREST ===
rf_imp = pd.Series(
    rf_model.feature_importances_,
    index=features_reg
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
rf_imp.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importance — Random Forest (drivers des ventes)')
ax.set_xlabel('Importance (MDI)')
plt.tight_layout()
plt.savefig('/tmp/rf_importance.png', dpi=100, bbox_inches='tight')
plt.show()

# Insight principal
top_driver = rf_imp.sort_values(ascending=False).index[0]
print(f"Principaux drivers de vente :")
print(rf_imp.sort_values(ascending=False).head(5))

In [ ]:
# === ANALYSE PARTIELLE — Effet du discount et du budget sur le CA ===
from sklearn.inspection import partial_dependence, PartialDependenceDisplay

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

features_pdp = ["AVG_DISCOUNT", "AVG_BUDGET"]
for i, feat in enumerate(features_pdp):
    feat_idx = features_reg.index(feat)
    pd_results = partial_dependence(rf_model, X3_train, features=[feat_idx], kind='average')
    ax[i].plot(pd_results['grid_values'][0], pd_results['average'][0], color='steelblue')
    ax[i].set_xlabel(feat)
    ax[i].set_ylabel('CA prédit (effet marginal)')
    ax[i].set_title(f'Dependance partielle — {feat}')
    ax[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/rf_pdp.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Récapitulatif des performances & insights marketing

In [ ]:
# === TABLEAU DE BORD RÉCAPITULATIF ===
print("="*60)
print("RÉCAPITULATIF DES MODÈLES ML MARKETING")
print("="*60)

print("\n[ Modèle 1 — K-Means Clustering ]")
print(f"  K optimal         : {K_FINAL} segments")
print(f"  Silhouette Score  : {max(sil_scores):.4f}")
print(f"  Table produite    : CUSTOMER_SEGMENTS")

print("\n[ Modèle 3 — Régression Linéaire ]")
print(f"  R²                : {r2:.4f}")
print(f"  RMSE              : {rmse:,.0f}")
print(f"  Table produite    : ML_PROMOTION_FEATURES")

print("\n[ Modèle 4 — Random Forest ]")
print(f"  R²                : {r2_rf:.4f}")
print(f"  RMSE              : {rmse_rf:,.0f}")
print(f"  Top driver ventes : {rf_imp.sort_values(ascending=False).index[0]}")

print("\n" + "="*60)
print("RECOMMANDATIONS MARKETING")
print("="*60)
print("1. Cibler les 'Chasseurs de promos' avec des offres flash à fort discount")
print("2. Activer les 'Dormants' via campagne email (budget modéré, forte personnalisation)")
print("3. Concentrer le budget campagne sur les régions à uplift positif")
print("4. Optimiser le taux de conversion comme levier n°1 de CA")

##### Vérification des tables créées

In [ ]:
SELECT 
    'CUSTOMER_SEGMENTS'      AS table_name, COUNT(*) AS nb_lignes FROM ANYCOMPANY_LAB.ANALYTICS.CUSTOMER_SEGMENTS
UNION ALL

SELECT 
    'ML_PROMOTION_FEATURES'  AS table_name, COUNT(*) AS nb_lignes FROM ANYCOMPANY_LAB.ANALYTICS.ML_PROMOTION_FEATURES
ORDER BY table_name;